# Beneish field gaps: fixed collection prefix

This executed companion recomputes summary tables from the **public aggregate receipt**. It needs no credentials or private data. Source coverage was audited separately through `OfflineDataReader` against an explicitly pinned, closed copy, with blocked network access and unchanged inputs.

The collection is still running: **503 securities from an ETF-holdings proxy**, not certified exact current S&P 500 membership. The 397 successful tasks represent a partial, potentially biased prefix. Counts describe source statement rows, not aligned annual observations or eligible scores. [Method and source caveats](../docs/beneish-field-gaps-2026-10-22.md).

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

repo = Path.cwd()
if not (repo / "docs").is_dir():
    repo = repo.parent
receipt = json.loads((repo / "docs/validation-beneish-field-gaps-2026-10-22.json").read_text())
audit = receipt["field_audit"]
groups = {(g["function"], g["table"]): g for g in audit["groups"]}
checkpoint = receipt["collection_checkpoint"]
assert sum(checkpoint["job_status_counts"].values()) == checkpoint["planned_jobs"]
assert audit["selection_count"] == checkpoint["job_status_counts"]["succeeded"]
buckets = ["column_absent_rows", "missing_marker_rows", "invalid_numeric_rows",
           "nonfinite_numeric_rows", "boolean_rows", "finite_rows"]
for group in groups.values():
    for field in group["fields"].values():
        assert sum(field[k] for k in buckets) == group["source_rows"]
        assert field["positive_rows"] + field["zero_rows"] + field["negative_rows"] == field["finite_rows"]
assert not audit["factor_calculated"] and not audit["research_ready"]
display(pd.DataFrame(list(checkpoint["job_status_counts"].items()), columns=["Task state", "Count"]))

,Task state,Count
0,succeeded,397
1,failed,40
2,inflight,1
3,pending,1071


## Blocking fields and nearby alternatives

Each fraction uses its own endpoint/table row denominator. Finite includes zero and negative values, so it does not measure domain eligibility. The alternatives in this table remain semantically unverified; D&A and DD&A do not establish pure depreciation.

In [2]:
candidates = [
    ("INCOME_STATEMENT", "depreciation", "Pure depreciation: all missing markers"),
    ("BALANCE_SHEET", "longTermDebtNoncurrent", "Noncurrent debt: all missing markers"),
    ("INCOME_STATEMENT", "incomeBeforeExtraordinaryItems", "IBEI probe: absent column"),
    ("INCOME_STATEMENT", "depreciationAndAmortization", "D&A: not pure depreciation"),
    ("CASH_FLOW", "depreciationDepletionAndAmortization", "DD&A: not pure depreciation"),
    ("BALANCE_SHEET", "longTermDebt", "Total LTD: conditional candidate"),
    ("BALANCE_SHEET", "currentLongTermDebt", "Current LTD: conditional candidate"),
    ("BALANCE_SHEET", "currentNetReceivables", "Broader receivables: trade scope unverified"),
]
rows = []
for function, name, interpretation in candidates:
    annual = groups[(function, "annual")]["fields"][name]
    quarterly = groups[(function, "quarterly")]["fields"][name]
    rows.append({
        "Provider field": name,
        "Annual finite / rows": f"{annual['finite_rows']:,} / {annual['source_rows']:,}",
        "Quarterly finite / rows": f"{quarterly['finite_rows']:,} / {quarterly['source_rows']:,}",
        "Annual symbols with any finite / selected": f"{annual['provider_symbols_with_finite']} / {annual['provider_symbol_denominator']}",
        "Interpretation": interpretation,
    })
display(pd.DataFrame(rows))

,Provider field,Annual finite / rows,Quarterly finite / rows,Annual symbols with any finite / selected,Interpretation
0,depreciation,"0 / 2,504","0 / 10,049",0 / 131,Pure depreciation: all missing markers
1,longTermDebtNoncurrent,"0 / 2,577","0 / 10,333",0 / 134,Noncurrent debt: all missing markers
2,incomeBeforeExtraordinaryItems,"0 / 2,504","0 / 10,049",0 / 131,IBEI probe: absent column
3,depreciationAndAmortization,"2,468 / 2,504","9,821 / 10,049",131 / 131,D&A: not pure depreciation
4,depreciationDepletionAndAmortization,"2,511 / 2,524","10,017 / 10,118",132 / 132,DD&A: not pure depreciation
5,longTermDebt,"2,347 / 2,577","9,361 / 10,333",134 / 134,Total LTD: conditional candidate
6,currentLongTermDebt,"1,316 / 2,577","4,474 / 10,333",133 / 134,Current LTD: conditional candidate
7,currentNetReceivables,"2,552 / 2,577","10,176 / 10,333",134 / 134,Broader receivables: trade scope unverified


## Conditional debt decomposition

`longTermDebt - currentLongTermDebt` is a candidate only when the two fields have consistent scope, dates, units and measurement. Negative results are a diagnostic, never floored or filled. Nonnegative results also need semantic evidence. This notebook does not create a standard debt field.

In [3]:
rows = []
for table in ("annual", "quarterly"):
    debt = groups[("BALANCE_SHEET", table)]["conditional_debt_diagnostic"]
    assert debt["jointly_finite_rows"] == debt["difference_nonnegative_rows"] + debt["difference_negative_rows"]
    assert not debt["standard_field_filled"]
    rows.append({"Table": table, "Source rows": debt["source_rows"],
                 "Jointly finite": debt["jointly_finite_rows"],
                 "Nonnegative difference": debt["difference_nonnegative_rows"],
                 "Negative difference": debt["difference_negative_rows"]})
display(pd.DataFrame(rows))

,Table,Source rows,Jointly finite,Nonnegative difference,Negative difference
0,annual,2577,1313,1211,102
1,quarterly,10333,4470,4108,362


## Endpoint coverage and metadata gates

Source labels and nonempty metadata do not authenticate annual duration, unit scale, issuer identity, reporting scope or publication timing. Failures and pending jobs are outside these denominators. Cross-statement coverage is not inferred from an endpoint having at least one numeric observation.

In [4]:
rows = []
for function in ("INCOME_STATEMENT", "BALANCE_SHEET", "CASH_FLOW"):
    g = groups[(function, "annual")]
    metadata = g["metadata_coverage"]
    assert metadata["amount_unit_status_counts"]["unverified"] == g["source_rows"]
    assert all(metadata[k]["nonempty_rows"] == 0 for k in ("asset_id", "available_at", "period_start", "fiscal_year"))
    rows.append({"Endpoint": function, "Selected symbols": g["provider_symbol_denominator"],
                 "Annual rows": g["source_rows"], "Quarterly rows": groups[(function, "quarterly")]["source_rows"],
                 "Verified unit rows": metadata["amount_unit_status_counts"]["verified"]})
display(pd.DataFrame(rows))

,Endpoint,Selected symbols,Annual rows,Quarterly rows,Verified unit rows
0,INCOME_STATEMENT,131,2504,10049,0
1,BALANCE_SHEET,134,2577,10333,0
2,CASH_FLOW,132,2524,10118,0


## Decision and next acceptance gates

All three direct blocking concepts have **zero finite source values**. Annual debt candidates are jointly finite in 1,313 rows, including 102 negative differences. Metadata is also incomplete. No M-score or eligible-pair count was calculated by this audit.

Finish or explicitly close the collection and classify failures; retain the proxy-universe label. First establish whether AV can supply pure depreciation, then review debt decomposition and IBEI/trade-receivable bridges. Keep strict Beneish blocked when a required concept cannot be established. Any alternative score needs its own named definition and validation. After issuer, annual-window, unit and scope evidence is reviewed, run the existing annual adapter on a closed selection. Market testing also needs authentic availability and historical universe evidence.

For an authorized local source rerun, use `examples/audit_beneish_field_coverage.py` with a **closed** data root and a private explicit symbol/function/snapshot selection. This public notebook checks and presents aggregate arithmetic; it cannot independently authenticate private source lineage.